In [69]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

In [2]:
!unzip -q agricultural-images.zip -d /content/dataset

In [42]:
!pip install split-folders
import splitfolders
splitfolders.ratio("Agricultural-crops", output="dataset_final", seed=42, ratio=(0.80, 0.20))

Copying files: 829 files [00:00, 2813.73 files/s]


In [43]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE)

cuda


In [85]:
weights = ResNet50_Weights.DEFAULT
val_preprocess = weights.transforms()
train_preprocess = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
print(preprocess)
model = resnet50(weights=weights).to(DEVICE)

ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [18]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [45]:
model.fc

Linear(in_features=2048, out_features=1000, bias=True)

In [86]:
train_dataset = torchvision.datasets.ImageFolder(
    'dataset_final/train',
    transform=train_preprocess
)
valid_dataset = torchvision.datasets.ImageFolder(
    'dataset_final/val',
    transform=val_preprocess
)

In [99]:
idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}

In [87]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

In [88]:
for param in model.parameters():
  param.requires_grad = False

In [89]:
model.fc.weight.requires_grad = True
model.fc.bias.requires_grad = True

In [90]:
model.fc = nn.Linear(in_features=2048, out_features=30, bias=True)
model = model.to(DEVICE)

In [59]:
for param in model.parameters():
  print(param.requires_grad)

False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
True
True


In [91]:
import numpy as np

def fit(
        model: torch.nn.Module,
        epochs: int,
        optimizer: torch.optim.Optimizer,
        train_loader: DataLoader,
        valid_loader: DataLoader
        ) -> tuple[list, ...]:

  train_epoch_acc = []
  train_epoch_losses = []
  valid_epoch_losses = []
  valid_epoch_acc =[]
  for epoch in range(epochs):
      model.train()
      loss_batch = []
      acc_batch  = []

      for images, labels in train_loader:
          images = images.to(DEVICE)
          labels = labels.to(DEVICE)

          preds = model(images).squeeze(-1)
          loss = criterion(preds, labels.long())
          loss_batch.append(loss.item())

          # поправлена асс для мультклассовой классификации
          predicted_classes = preds.argmax(dim=1)
          accuracy = (predicted_classes == labels).float().mean().item()
          acc_batch.append(accuracy)

          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

      train_epoch_losses.append(np.mean(loss_batch))
      train_epoch_acc.append(np.mean(acc_batch))

      model.eval()
      loss_batch = []
      acc_batch  = []
      for images, labels in valid_loader:
          images = images.to(DEVICE)
          labels = labels.to(DEVICE)
          with torch.no_grad():
              preds = model(images).squeeze(-1)

          loss = criterion(preds, labels.long())
          loss_batch.append(loss.item())

          # поправлена асс для мультклассовой классификации
          predicted_classes = preds.argmax(dim=1)
          accuracy = (predicted_classes == labels).float().mean().item()
          acc_batch.append(accuracy)

      valid_epoch_losses.append(np.mean(loss_batch))
      valid_epoch_acc.append(np.mean(acc_batch))

      print(f'Epoch: {epoch}  loss_train: {train_epoch_losses[-1]:.3f}, loss_valid: {valid_epoch_losses[-1]:.3f}')
      print(f'\t  metrics_train: {train_epoch_acc[-1]:.3f}, metrics_valid: {valid_epoch_acc[-1]:.3f}')
  return train_epoch_losses, valid_epoch_losses, train_epoch_acc, valid_epoch_acc

In [92]:
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001, weight_decay=0.1)
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

In [95]:
train_loss, valid_loss, train_metric, valid_metric = fit(
     epochs=20,
     model=model,
     optimizer=optimizer,
     train_loader=train_loader,
     valid_loader=valid_loader
 )

Epoch: 0  loss_train: 1.807, loss_valid: 2.083
	  metrics_train: 0.860, metrics_valid: 0.666
Epoch: 1  loss_train: 1.792, loss_valid: 2.077
	  metrics_train: 0.849, metrics_valid: 0.681
Epoch: 2  loss_train: 1.762, loss_valid: 2.044
	  metrics_train: 0.849, metrics_valid: 0.692
Epoch: 3  loss_train: 1.752, loss_valid: 2.024
	  metrics_train: 0.868, metrics_valid: 0.677
Epoch: 4  loss_train: 1.743, loss_valid: 2.014
	  metrics_train: 0.865, metrics_valid: 0.687
Epoch: 5  loss_train: 1.739, loss_valid: 2.023
	  metrics_train: 0.847, metrics_valid: 0.687
Epoch: 6  loss_train: 1.722, loss_valid: 1.996
	  metrics_train: 0.851, metrics_valid: 0.666
Epoch: 7  loss_train: 1.698, loss_valid: 2.004
	  metrics_train: 0.864, metrics_valid: 0.687
Epoch: 8  loss_train: 1.702, loss_valid: 1.977
	  metrics_train: 0.878, metrics_valid: 0.692
Epoch: 9  loss_train: 1.682, loss_valid: 1.974
	  metrics_train: 0.866, metrics_valid: 0.692
Epoch: 10  loss_train: 1.646, loss_valid: 1.961
	  metrics_train: 0.88

In [ ]:
train_loss, valid_loss, train_metric, valid_metric = ftrain_loss, fvalid_loss, ftrain_metric, fvalid_metric

In [96]:
torch.save(model.state_dict(), 'plants_resnet50_final.pth')

In [97]:
import requests
from io import BytesIO
from PIL import Image

def get_prediction(path: str) -> str:
  if path.startswith('http'):
        response = requests.get(path)
        img = Image.open(BytesIO(response.content)).convert('RGB')
  else:
        img = Image.open(path).convert('RGB')

  tensor= preprocess(img).unsqueeze(0).to(DEVICE)

  model.eval()
  with torch.no_grad():
    output = model(tensor)
    pred = output.argmax().item()

  class_name = idx_to_class.get(pred, 'Unknown')

  return class_name

In [100]:
get_prediction('https://newflora.ru/img/123/resize:fill:363:267:0:0/plain/https://newflora.ru/wp-content/uploads/2024/07/banana-transformed.png@jpg')

'banana'

In [104]:
from getpass import getpass

# 1. Данные владельца репозитория (у кого ты коллаборатор)
OWNER = "silentl0tus"
REPO = "phase2_week1"

# 2. Твой ник и твой токен (PAT)
MY_USER = "Ali-B-Indahouse"
TOKEN = getpass("Вставь СВОЙ GitHub Token: ")

# 3. Клонируем через твой токен в чужой репо
repo_url = f"https://{TOKEN}@github.com/{OWNER}/{REPO}.git"
!git clone {repo_url}

Вставь СВОЙ GitHub Token: ··········
Cloning into 'phase2_week1'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 19 (delta 5), reused 9 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 8.91 KiB | 8.91 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [105]:
!ls

Agricultural-crops	 dataset	phase2_week1
agricultural-images.zip  dataset_final	sample_data


In [106]:
!mv "resnet50.ipynb" "phase2_week1/notebooks/"

mv: cannot stat 'resnet50.ipynb': No such file or directory
